# Experiment: TRUMP Trader Archetypes Analysis

This notebook analyzes three simple trader types that primarily trade TRUMP coin:
- rational arbitrageurs
- manipulators
- herd-following retail

The notebook focuses on two outputs:
- Shapley values for all available TRUMP market features for each trader type
- top-3 feature explanations for each trader type


In [1]:
import sys
from pathlib import Path
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "trader_archetypes.py").exists():
    repo_root = repo_root.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from trader_archetypes import (
    TRADER_ORDER,
    notebook_summary_bundle,
    run_smoke_tests,
    top_features_for_trader,
)


## Validation

The first step is to make sure the notebook-facing implementation loads and all
core pieces execute.


In [2]:
tests = run_smoke_tests(dataset="trump")
display(tests)
assert tests["passed"].all(), "One or more smoke tests failed."


,test,passed,detail
0,dataset_loads,True,"rows=9650, features=9"
1,trader_profiles_build,True,profile_columns=6
2,shapley_summary_builds,True,rows=27
3,action_summary_builds,True,rows=9


## Build The Analysis Bundle

This loads the TRUMP feature set, computes trader scores and actions, and
builds the Shapley summary plus top-feature explanations.


In [3]:
bundle = notebook_summary_bundle(dataset="trump", shapley_sample_size=32, random_state=7)

print("Trader profile preview:")
display(bundle["profiles"].head())
print("Action summary:")
display(bundle["action_summary"])


Trader profile preview:


,rational_arbitrageur_score,rational_arbitrageur_action,manipulator_score,manipulator_action,herd_retail_score,herd_retail_action
Datetime,,,,,,
2025-01-25 12:00:00+00:00,0.948376,buy_mispricing,-0.598858,fade_or_dump,-0.997798,panic_sell
2025-01-25 13:00:00+00:00,-0.997503,sell_rally,0.999936,pump_or_front_run,0.997231,chase_uptrend
2025-01-25 14:00:00+00:00,-0.754987,sell_rally,0.587733,pump_or_front_run,-0.678372,panic_sell
2025-01-25 15:00:00+00:00,-0.648469,sell_rally,0.993339,pump_or_front_run,0.432145,chase_uptrend
2025-01-25 16:00:00+00:00,-0.901982,sell_rally,0.999329,pump_or_front_run,0.924265,chase_uptrend


Action summary:


,trader,display_name,action,count,share
0,herd_retail,Herd-Following Retail,panic_sell,4745,0.491710
1,herd_retail,Herd-Following Retail,chase_uptrend,3559,0.368808
2,herd_retail,Herd-Following Retail,hold,1346,0.139482
3,manipulator,Manipulator,fade_or_dump,4551,0.471606
4,manipulator,Manipulator,pump_or_front_run,4138,0.428808
5,manipulator,Manipulator,hold,961,0.099585
6,rational_arbitrageur,Rational Arbitrageur,buy_mispricing,4236,0.438964
7,rational_arbitrageur,Rational Arbitrageur,sell_rally,3420,0.354404
8,rational_arbitrageur,Rational Arbitrageur,hold,1994,0.206632


## Top 3 Features Per Trader

These tables rank features by mean absolute Shapley value. The text underneath
turns those rankings into short explanations.


In [4]:
for trader_name in TRADER_ORDER:
    print()
    print(f"=== {trader_name} ===")
    display(top_features_for_trader(bundle["shapley_summary"], trader_name, top_n=3))
    for line in bundle["explanations"][trader_name]:
        print("-", line)



=== rational_arbitrageur ===


,trader,display_name,feature,feature_label,mean_shapley,mean_abs_shapley,rank
18,rational_arbitrageur,Rational Arbitrageur,feature_log_return_1h,1h Return,0.000492,0.437413,1.0
19,rational_arbitrageur,Rational Arbitrageur,feature_log_return_4h,4h Return,-0.027040,0.292954,2.0
20,rational_arbitrageur,Rational Arbitrageur,feature_volatility_24h,24h Volatility,0.035545,0.173202,3.0


- 1h Return matters because it captures very recent momentum; on average it pushes the trader toward more aggressive action for rational arbitrageur.
- 4h Return matters because it captures short-horizon trend; on average it pushes the trader toward restraint or reversal for rational arbitrageur.
- 24h Volatility matters because it captures market stress and risk; on average it pushes the trader toward more aggressive action for rational arbitrageur.

=== manipulator ===


,trader,display_name,feature,feature_label,mean_shapley,mean_abs_shapley,rank
9,manipulator,Manipulator,feature_log_volume,Log Volume,-0.089538,0.706723,1.0
10,manipulator,Manipulator,feature_log_return_1h,1h Return,-0.022915,0.281017,2.0
11,manipulator,Manipulator,feature_volatility_24h,24h Volatility,-0.052148,0.180457,3.0


- Log Volume matters because it captures market attention and activity; on average it pushes the trader toward restraint or reversal for manipulator.
- 1h Return matters because it captures very recent momentum; on average it pushes the trader toward restraint or reversal for manipulator.
- 24h Volatility matters because it captures market stress and risk; on average it pushes the trader toward restraint or reversal for manipulator.

=== herd_retail ===


,trader,display_name,feature,feature_label,mean_shapley,mean_abs_shapley,rank
0,herd_retail,Herd-Following Retail,feature_log_volume,Log Volume,-0.043105,0.488929,1.0
1,herd_retail,Herd-Following Retail,feature_log_return_1h,1h Return,-0.015211,0.375825,2.0
2,herd_retail,Herd-Following Retail,feature_log_return_4h,4h Return,0.021418,0.278338,3.0


- Log Volume matters because it captures market attention and activity; on average it pushes the trader toward restraint or reversal for herd-following retail.
- 1h Return matters because it captures very recent momentum; on average it pushes the trader toward restraint or reversal for herd-following retail.
- 4h Return matters because it captures short-horizon trend; on average it pushes the trader toward more aggressive action for herd-following retail.


## Full Shapley Summary

This table includes all features for all three TRUMP trader archetypes.


In [5]:
display(bundle["shapley_summary"])


,trader,display_name,feature,feature_label,mean_shapley,mean_abs_shapley,rank
0,herd_retail,Herd-Following Retail,feature_log_volume,Log Volume,-0.043105,0.488929,1.0
1,herd_retail,Herd-Following Retail,feature_log_return_1h,1h Return,-0.015211,0.375825,2.0
2,herd_retail,Herd-Following Retail,feature_log_return_4h,4h Return,0.021418,0.278338,3.0
3,herd_retail,Herd-Following Retail,feature_log_return_24h,24h Return,-0.009550,0.148639,4.0
4,herd_retail,Herd-Following Retail,feature_day_sin,Day Sin,0.025590,0.088280,5.0
5,herd_retail,Herd-Following Retail,feature_volatility_24h,24h Volatility,-0.011934,0.064645,6.0
6,herd_retail,Herd-Following Retail,feature_day_cos,Day Cos,0.000000,0.000000,7.0
7,herd_retail,Herd-Following Retail,feature_hour_cos,Hour Cos,0.000000,0.000000,7.0
8,herd_retail,Herd-Following Retail,feature_hour_sin,Hour Sin,0.000000,0.000000,7.0
9,manipulator,Manipulator,feature_log_volume,Log Volume,-0.089538,0.706723,1.0


## Notes

These trader types are simple, hand-crafted TRUMP trading archetypes. The
analysis explains their decision scores over the current feature table; it
does not claim to model full market impact or strategic equilibrium.
